# Paso 2 — Búsqueda de hiperparámetros con Optuna



In [ ]:
import os
import platform
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data 
from torch_geometric.transforms import RandomLinkSplit
import time
import pandas as pd
import numpy as np
import random
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as ov 
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
BASE_OUTPUT_DIR = './output' 
DATA_FILENAME = 'processed_graph_data.pt'
METADATA_FILENAME = 'metadata.pt'

data_path = os.path.join(BASE_OUTPUT_DIR, DATA_FILENAME)
metadata_path = os.path.join(BASE_OUTPUT_DIR, METADATA_FILENAME)

data = None
IN_CHANNELS = None
try:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    map_loc = 'cpu' if device.type == 'cpu' else None

    print(f"Dispositivo de Cómputo Actual: {device}")
    print(f"Mapeo de carga: {map_loc if map_loc else 'Original'}")

    data = torch.load(data_path, map_location=map_loc, weights_only=False) 
    
    # Cargar metadata: USAR weights_only=False
    metadata = torch.load(metadata_path, weights_only=False) 
    
    IN_CHANNELS = data.x.shape[1]
    
    print(f"✔️ Grafo y Metadata cargados exitosamente. IN_CHANNELS: {IN_CHANNELS}")
    print(f"Grafo movido/verificado en el dispositivo: {device}")
    
except Exception as e:
    # Si todo falla, imprime el error y establece las variables a None
    print(f"🚨 ERROR FATAL al cargar los archivos. Mensaje: {e}")
    data = None
    IN_CHANNELS = None

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

### model architecture

In [ ]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_heads=1, add_self_loops=True, dropout_rate=0.0, activation_fn_name="relu"):
        super(GNNEncoder, self).__init__()
        
        self.conv1 = GATConv(in_channels, hidden_channels, heads=num_heads, 
                              add_self_loops=add_self_loops, edge_dim=1, dropout=dropout_rate)
        
        self.conv2 = GATConv(hidden_channels * num_heads, out_channels, heads=1, 
                              add_self_loops=add_self_loops, edge_dim=1, concat=False, dropout=dropout_rate)
        
        self.dropout_layer = nn.Dropout(dropout_rate) 

        if activation_fn_name == "relu":
            self.activation_fn = F.relu
        elif activation_fn_name == "tanh":
            self.activation_fn = F.tanh
        else:
            raise ValueError(f"Función de activación '{activation_fn_name}' no soportada.")

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = self.activation_fn(x)  
        x = self.dropout_layer(x)  
        
        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        return x

class LinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(LinkPredictor, self).__init__()
        self.lin1 = nn.Linear(in_channels * 2, hidden_channels) 
        self.lin2 = nn.Linear(hidden_channels, out_channels) 

    def forward(self, x_i, x_j):
        x = torch.cat([x_i, x_j], dim=-1) 
        x = self.lin1(x)
        x = F.relu(x) 
        x = self.lin2(x)
        return x

print("✔️ Clases GNNEncoder y LinkPredictor definidas.")

## Training evaluation

In [ ]:
def train(model, predictor, data, optimizer, criterion):
    model.train()
    predictor.train()
    optimizer.zero_grad()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Predicción para enlaces positivos de entrenamiento 
    pos_edge_index = data.train_pos_edge_index
    pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])

    # Predicción para enlaces negativos de entrenamiento
    neg_edge_index = data.train_neg_edge_index
    neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])

    # Concatenar predicciones y etiquetas reales
    pred = torch.cat([pos_pred, neg_pred], dim=0)
    target = torch.cat([torch.ones(pos_pred.size(0), device=pred.device), torch.zeros(neg_pred.size(0), device=pred.device)], dim=0)

    # Pérdida y actualización de pesos
    train_loss = criterion(pred.squeeze(), target)
    train_loss.backward()
    optimizer.step()

    # Calcular métricas (en CPU)
    pred_cpu = pred.detach().cpu().numpy().squeeze()
    target_cpu = target.cpu().numpy()
    preds_bin = (pred_cpu >= 0.0).astype(int) # Umbral de 0.0 para logits

    train_auc = roc_auc_score(target_cpu, pred_cpu)
    train_acc = accuracy_score(target_cpu, preds_bin)
    train_precision = precision_score(target_cpu, preds_bin, zero_division=0)
    train_recall = recall_score(target_cpu, preds_bin, zero_division=0)
    train_f1 = f1_score(target_cpu, preds_bin, zero_division=0)

    return train_loss.item(), train_auc, train_acc, train_precision, train_recall, train_f1

@torch.no_grad() 
def test(model, predictor, data):
    model.eval()
    predictor.eval()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Función auxiliar para obtener predicciones y etiquetas
    def get_preds_targets(pos_edge_index, neg_edge_index):
        pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
        neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])
        
        preds_tensor = torch.cat([pos_pred, neg_pred], dim=0).squeeze()
        targets_tensor = torch.cat([torch.ones(pos_pred.size(0), device=preds_tensor.device), torch.zeros(neg_pred.size(0), device=preds_tensor.device)], dim=0)
        
        return preds_tensor.cpu().numpy(), targets_tensor.cpu().numpy()

    # Validación
    val_preds, val_targets = get_preds_targets(data.val_pos_edge_index, data.val_neg_edge_index)
    
    # Prueba
    test_preds, test_targets = get_preds_targets(data.test_pos_edge_index, data.test_neg_edge_index)

    # Loss (Usando tensores en CPU para BCEWithLogitsLoss)
    val_loss = F.binary_cross_entropy_with_logits(torch.tensor(val_preds), torch.tensor(val_targets))
    test_loss = F.binary_cross_entropy_with_logits(torch.tensor(test_preds), torch.tensor(test_targets))

    # Calcular Métricas
    val_auc = roc_auc_score(val_targets, val_preds) 
    test_auc = roc_auc_score(test_targets, test_preds) 

    val_preds_bin = (val_preds >= 0.0).astype(int) 
    test_preds_bin = (test_preds >= 0.0).astype(int)

    val_acc = accuracy_score(val_targets, val_preds_bin)
    test_acc = accuracy_score(test_targets, test_preds_bin)
    val_precision = precision_score(val_targets, val_preds_bin, zero_division=0)
    test_precision = precision_score(test_targets, test_preds_bin, zero_division=0)
    val_recall = recall_score(val_targets, val_preds_bin, zero_division=0)
    test_recall = recall_score(test_targets, test_preds_bin, zero_division=0)
    val_f1 = f1_score(val_targets, val_preds_bin, zero_division=0)
    test_f1 = f1_score(test_targets, test_preds_bin, zero_division=0)

    return val_loss, test_loss, val_auc, test_auc, val_acc, test_acc, val_precision, test_precision, val_recall, test_recall, val_f1, test_f1

print("✔️ Funciones de entrenamiento y evaluación definidas.")

## División de enlaces

In [ ]:
if data is not None:
    set_seed(42) 
    print("\nDividiendo enlaces para entrenamiento/validación/prueba (predicción de enlaces)...")
    

    transform = RandomLinkSplit(
        num_val=0.1,
        num_test=0.1,
        is_undirected=True,
        add_negative_train_samples=True,
        split_labels=True
    )
    
    train_data, val_data, test_data = transform(data)

    # Crear el objeto data_for_tuning con los índices de TRAIN
    data_for_tuning = Data(x=train_data.x, edge_index=train_data.edge_index, edge_attr=train_data.edge_attr)
    data_for_tuning.train_pos_edge_index = train_data.pos_edge_label_index
    data_for_tuning.train_neg_edge_index = train_data.neg_edge_label_index
    data_for_tuning.val_pos_edge_index   = val_data.pos_edge_label_index
    data_for_tuning.val_neg_edge_index   = val_data.neg_edge_label_index
    data_for_tuning.test_pos_edge_index  = test_data.pos_edge_label_index
    data_for_tuning.test_neg_edge_index  = test_data.neg_edge_label_index
    
    data_for_tuning = data_for_tuning.to(device)
    
    print(f"  Total de Enlaces Positivos de Entrenamiento: {data_for_tuning.train_pos_edge_index.shape[1]}")
    
    preprocessed_data_split = (data_for_tuning, IN_CHANNELS)
    print("✔️ División de datos completada y lista para Optuna.")


## Ejecución Optuna

In [ ]:
def objective(trial, preprocessed_data_split):
    # Usamos la semilla para la reproducibilidad DENTRO de cada trial.
    set_seed(42) 

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, in_channels_computed = preprocessed_data_split
    data = data.to(device) 

    # --- Sugerir Hiperparámetros con Optuna ---
    hidden_channels = trial.suggest_categorical("hidden_channels", [64, 128, 256])
    out_channels = trial.suggest_categorical("out_channels", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    predictor_hidden_channels = trial.suggest_categorical("predictor_hidden_channels", [32, 64, 128])
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5) 
    activation_function_name = trial.suggest_categorical("activation_function", ["relu", "tanh"])

    # El número de épocas 
    epochs_for_trial = trial.suggest_int("epochs", 50, 200, step=50)

    # --- Inicialización del Modelo ---
    model = GNNEncoder(in_channels_computed, hidden_channels, out_channels, 
                       num_heads=num_heads, dropout_rate=dropout_rate, 
                       activation_fn_name=activation_function_name).to(device)
    predictor = LinkPredictor(out_channels, predictor_hidden_channels, 1).to(device)

    optimizer = torch.optim.Adam(list(model.parameters()) + list(predictor.parameters()), lr=learning_rate)
    criterion = torch.nn.BCEWithLogitsLoss()

    # --- Entrenamiento y Evaluación ---
    print(f"\n--- Trial {trial.number}: HPs: {trial.params} ---")
    best_val_auc = 0.0
    best_val_auc_epoch_metrics = {} 

    for epoch in range(1, epochs_for_trial + 1):
        train_loss, train_auc, train_acc, train_precision, train_recall, train_f1 = train(model, predictor, data, optimizer, criterion)
        val_loss, test_loss, val_auc, test_auc, val_acc, test_acc, val_precision, test_precision, val_recall, test_recall, val_f1, test_f1 = test(model, predictor, data)
        
        trial.report(val_auc, epoch)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_val_auc_epoch_metrics = {
                'epoch_at_best_val_auc': epoch,
                'train_loss': train_loss, 'val_loss': val_loss.item(), 'test_loss': test_loss.item(),
                'train_auc': train_auc, 'val_auc': val_auc, 'test_auc': test_auc,
                'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc,
                'train_precision': train_precision, 'val_precision': val_precision, 'test_precision': test_precision,
                'train_recall': train_recall, 'val_recall': val_recall, 'test_recall': test_recall,
                'train_f1': train_f1, 'val_f1': val_f1, 'test_f1': test_f1,
            }

        if trial.should_prune():
            print(f"Trial {trial.number} podado en la época {epoch}.")
            if best_val_auc_epoch_metrics:
                trial.set_user_attr("best_val_auc_epoch_metrics", best_val_auc_epoch_metrics)
            raise optuna.exceptions.TrialPruned()

        if epoch % 10 == 0 or epoch == epochs_for_trial:
            print(f'  Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}')

    trial.set_user_attr("best_val_auc_epoch_metrics", best_val_auc_epoch_metrics)
    
    return best_val_auc

In [ ]:
if 'preprocessed_data_split' in locals() and preprocessed_data_split[0] is not None:
    
    start_total_time = time.time()
    set_seed(42) 
    
    print("\n--- Información del Sistema ---")
    print(f"Sistema Operativo: {platform.system()} {platform.release()} ({platform.version()})")
    print(f"Arquitectura: {platform.machine()}")
    print(f"Núcleos de CPU (físicos/lógicos): {psutil.cpu_count(logical=False)}/{os.cpu_count()}")
    total_ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"Memoria RAM Total: {total_ram_gb:.2f} GB")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        print(f"GPU Disponible: Sí")
        print(f"Nombre de GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memoria GPU Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("GPU Disponible: No (Usando CPU)")
    print(f"Dispositivo de Cómputo: {device}")

    print("\n--- Ejecutando Búsqueda de Hiperparámetros con Optuna ---")
    
    # --- Guardar descripción del espacio de búsqueda (Fidelidad a tu código) ---
    search_space_description = f"""
    Espacio de Búsqueda de Hiperparámetros (Optuna):
    -----------------------------------------------
    - hidden_channels: {['64', '128', '256']} (Categórico)
    - out_channels: {['32', '64', '128']} (Categórico)
    - num_heads: {['2', '4', '8']} (Categórico)
    - learning_rate: [1e-4, 1e-2] (Log-uniforme)
    - predictor_hidden_channels: {['32', '64', '128']} (Categórico)
    - dropout_rate: [0.0, 0.5] (Uniforme)
    - activation_function: {['relu', 'tanh']} (Categórico)
    - epochs: [50, 200] con paso de 50 (Entero)
    """
    search_space_path = os.path.join(BASE_OUTPUT_DIR, "optuna_search_space.txt")
    with open(search_space_path, "w") as f:
        f.write(search_space_description)
    print(f"\nDescripción del espacio de búsqueda guardada en: {search_space_path}")


    # Configurar el estudio de Optuna
    study = optuna.create_study(
        direction="maximize",
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=1),
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    # Ejecutar la optimización
    n_trials = 20 # Ajusta esto a tu necesidad
    print(f"\nIniciando {n_trials} trials de búsqueda de hiperparámetros...")
    study.optimize(lambda trial: objective(trial, preprocessed_data_split), 
                   n_trials=n_trials, 
                   show_progress_bar=True)

    print("\n--- Búsqueda de Hiperparámetros Completada ---")
    print(f"Mejor trial:")
    print(f"  Valor (AUC de validación): {study.best_value:.4f}")
    print(f"  Mejores Hiperparámetros: {study.best_params}")

    # --- Guardado de Resultados Completo ---
    study_results_path = os.path.join(BASE_OUTPUT_DIR, "optuna_study_results.csv")
    df_results = study.trials_dataframe()
    df_results.to_csv(study_results_path, index=False)
    print(f"\nResultados completos del estudio (incluyendo HPs y estado) guardados en: {study_results_path}")

    # --- Generar visualizaciones de Optuna ---
    if 'ov' in locals():
        print("\n--- Generando visualizaciones de Optuna (requiere Plotly) ---")
        try:
            fig_history = ov.plot_optimization_history(study)
            fig_history.write_html(os.path.join(BASE_OUTPUT_DIR, "optuna_optimization_history.html"))
            print(f"Historia de optimización guardada en: {os.path.join(BASE_OUTPUT_DIR, 'optuna_optimization_history.html')}")
            
            fig_importance = ov.plot_param_importances(study)
            fig_importance.write_html(os.path.join(BASE_OUTPUT_DIR, "optuna_param_importances.html"))
            print(f"Importancia de parámetros guardada en: {os.path.join(BASE_OUTPUT_DIR, 'optuna_param_importances.html')}")
            
            fig_parallel = ov.plot_parallel_coordinate(study)
            fig_parallel.write_html(os.path.join(BASE_OUTPUT_DIR, "optuna_parallel_coordinate.html"))
            print(f"Gráfico de coordenadas paralelas guardado en: {os.path.join(BASE_OUTPUT_DIR, 'optuna_parallel_coordinate.html')}")

        except Exception as e:
            print(f"No se pudieron generar las visualizaciones de Optuna. Asegúrate de que 'plotly' esté instalado (`pip install plotly`). Error: {e}")

     # --- Guardar las 10 mejores arquitecturas basadas en métricas de TEST ---
    print("\n--- Guardando las 10 mejores arquitecturas basadas en Test AUC ---")
    completed_trials = []
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE:
            # Recuperar los hiperparámetros y las métricas guardadas
            hparams = trial.params
            metrics = trial.user_attrs.get("best_val_auc_epoch_metrics", {})
            
            # Combinar HPs y métricas
            trial_data = {**hparams, **metrics}
            trial_data['trial_id'] = trial.number
            trial_data['value_optimized_by_optuna'] = trial.value # El best_val_auc
            completed_trials.append(trial_data)
    
    if completed_trials:
        df_all_completed_trials = pd.DataFrame(completed_trials)
        # Ordenar por 'test_auc' de forma descendente y tomar los top 10
        # Puedes cambiar 'test_auc' por 'test_f1' si prefieres esa métrica
        top_10_architectures = df_all_completed_trials.sort_values(by='test_auc', ascending=False).head(10)
        
        top_10_path = os.path.join(BASE_OUTPUT_DIR, "top_10_architectures_test_metrics.csv")
        top_10_architectures.to_csv(top_10_path, index=False)
        print(f"Las 10 mejores arquitecturas (basadas en Test AUC) guardadas en: {top_10_path}")
        print("\nColumnas del archivo de las 10 mejores arquitecturas:")
        print(top_10_architectures.columns.tolist())
    else:
        print("No hay trials completados para generar el archivo de las 10 mejores arquitecturas.")


    end_total_time = time.time()
    total_execution_time = end_total_time - start_total_time
    print(f"\n✅ Tiempo total de ejecución del pipeline de Optuna: {total_execution_time:.2f} segundos ({total_execution_time/60:.2f} minutos)")

else:
    print("🚨 ERROR: No se puede ejecutar Optuna. 'preprocessed_data_split' no contiene datos válidos.")